# Cloning the Repo

In [1]:
#!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection

[WinError 2] Impossibile trovare il file specificato: 'Continual-hate-speech-detection'
c:\Uni\Continual-hate-speech-detection


# Install all the necessary library

In [2]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cella 1
import torch
print(torch.__version__)           # 2.6.0+cu128
print(torch.cuda.is_available())   # True
print(torch.cuda.get_device_properties(0).major)  # 12

2.7.0+cu128
True
12


In [4]:
# Cella 2 — solo se cella 1 passa
from transformers import AutoTokenizer

c:\Uni\Continual-hate-speech-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Cella 3 — solo se cella 2 passa
from model_utils.model_builder import CustomClassifier

In [6]:
from dataset.df_loader import getdf_davidson, getdf_hatexplain
from dataset.stream_generator import create_continual_stream, create_stream, online_stream#from transformers import (AutoTokenizer, AutoConfig)
from model_utils.trainer import ContinualTrainer

## Check if a GPU is avaible

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

DEVICE: cuda


### Getting the datasets

In [ ]:
print("Loading datasets...")

train_dv, val_dv = getdf_davidson()
train_hx, val_hx = getdf_hatexplain()

print(f"Davidson   — train: {train_dv.shape} | val: {val_dv.shape}")
print(f"HateXplain — train: {train_hx.shape} | val: {val_hx.shape}")

### Creating the datastream

In [ ]:
# stream di train per il continual learning
stream_davidson   = create_continual_stream([train_dv], batch_size=10)
stream_hatexplain = create_continual_stream([train_hx], batch_size=10)
full_stream       = stream_davidson + stream_hatexplain

# stream di val per evaluation (no shuffle)
val_stream_dv = create_stream(val_dv, batch_size=10, shuffle=False)
val_stream_hx = create_stream(val_hx, batch_size=10, shuffle=False)

### Creating the tokenizer and the pretrained model + custom head

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")

model = CustomClassifier(
    model_name="distilbert/distilroberta-base",
    num_labels=3,
    )

model.to(device)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


CustomClassifier(
  (backbone): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm):

## Training Session

In [ ]:
import matplotlib.pyplot as plt
from strategy.replay import ReplayStrategy
from strategy.distillation import DistillationStrategy
from strategy.ewc import EWCStrategy
from utils.metrics import compute_accuracy, compute_f1
from utils.plot_utils import (
    plot_loss,
    plot_batch_metrics,
    plot_conf_matrix,
    plot_classwise_accuracy,
    plot_label_distribution,
    plot_strategy_comparison,
    plot_forgetting
)

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────

BATCH_SIZE = 32
BUFFER_SIZE = 750
LR = 2e-5

print(f"Config: BATCH_SIZE={BATCH_SIZE}, BUFFER_SIZE={BUFFER_SIZE}, LR={LR}")
label_map = {"hatespeech": 0, "offensive": 1, "normal": 2}

# ─── STRATEGY ────────────────────────────────────────────────────────────────

#strategy = ReplayStrategy(buffer_size=BUFFER_SIZE)
#strategy = EWCStrategy(lambda_=0.4)
strategy = DistillationStrategy(alpha=0.5, temperature=2.0)

# ─── TRAINER ─────────────────────────────────────────────────────────────────

trainer = ContinualTrainer(model=model, tokenizer=tokenizer, device=device, lr=LR)
trainer.add_strategy(strategy)

# ─── TRAIN TASK 1 — Davidson ─────────────────────────────────────────────────

print("=== Task 1: Davidson ===")
losses_dv, preds_dv, labels_dv = trainer.train_continual(stream_davidson, label_map)

# eval su val set dopo task 1 — baseline pre-forgetting
preds_dv_pre, labels_dv_pre = trainer.evaluate(val_stream_dv, label_map)
acc_dv_pre = compute_accuracy(preds_dv_pre, labels_dv_pre)
f1_dv_pre  = compute_f1(preds_dv_pre, labels_dv_pre)
print(f"Davidson val (pre) — Acc: {acc_dv_pre:.4f} | F1: {f1_dv_pre:.4f}")

trainer.on_task_end(stream_davidson, label_map)

# ─── TRAIN TASK 2 — HateXplain ───────────────────────────────────────────────

print("\n=== Task 2: HateXplain ===")
losses_hx, preds_hx, labels_hx = trainer.train_continual(stream_hatexplain, label_map)
trainer.on_task_end(stream_hatexplain, label_map)

preds_hx_post, labels_hx_post = trainer.evaluate(val_stream_hx, label_map)
acc_hx_post = compute_accuracy(preds_hx_post, labels_hx_post)
f1_hx_post  = compute_f1(preds_hx_post, labels_hx_post)
print(f"HateXplain val     — Acc: {acc_hx_post:.4f} | F1: {f1_hx_post:.4f}")

# ─── EVAL FORGETTING ─────────────────────────────────────────────────────────

print("\n=== Eval forgetting ===")
preds_dv_post, labels_dv_post = trainer.evaluate(val_stream_dv, label_map)
acc_dv_post = compute_accuracy(preds_dv_post, labels_dv_post)
f1_dv_post  = compute_f1(preds_dv_post, labels_dv_post)
print(f"Davidson val (post) — Acc: {acc_dv_post:.4f} | F1: {f1_dv_post:.4f}")
print(f"Forgetting F1      : {f1_dv_pre - f1_dv_post:+.4f}")

# ─── PLOT ────────────────────────────────────────────────────────────────────

plot_loss(losses_dv + losses_hx)
plot_conf_matrix(preds_dv_post, labels_dv_post, label_map)
plot_conf_matrix(preds_hx_post, labels_hx_post, label_map)
plot_classwise_accuracy(
    torch.cat([preds_dv_post, preds_hx_post]),
    torch.cat([labels_dv_post, labels_hx_post]),
    label_map, batch_size=BATCH_SIZE
)
plot_label_distribution(train_dv, train_hx, label_map)
plot_forgetting(
    {"davidson":   [acc_dv_pre, acc_dv_post],
     "hatexplain": [None,       acc_hx_post]},
    label_map
)

In [ ]:
SAVE_DIR = "./models"

torch.save({
    "model_state_dict": model.state_dict(),
    "label_map": label_map,
    "strategy": strategy.__class__.__name__,
    "batch_size": BATCH_SIZE,
    "buffer_size": BUFFER_SIZE,
    "lr": LR
}, f"{SAVE_DIR}/model.pt")

tokenizer.save_pretrained(SAVE_DIR)

print("Salvataggio completato")

In [9]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load("./models/model.pt", map_location=DEVICE)

# -----------------------------
# Load label map
# -----------------------------
label_map = checkpoint["label_map"]

# invert map
id2label = {v: k for k, v in label_map.items()}

# -----------------------------
# Load tokenizer
# -----------------------------
loaded_tokenizer = AutoTokenizer.from_pretrained("./models")

# -----------------------------
# Rebuild model
# IMPORTANT:
# stesso backbone del training
# -----------------------------
loaded_model = CustomClassifier(
    model_name="distilbert-base-uncased",
    num_labels=len(label_map)
)

# -----------------------------
# Load trained weights
# -----------------------------
loaded_model.load_state_dict(checkpoint["model_state_dict"])

loaded_model.to(DEVICE)
loaded_model.eval()

print("Modello ricaricato correttamente")

RuntimeError: Error(s) in loading state_dict for CustomClassifier:
	Missing key(s) in state_dict: "backbone.transformer.layer.0.attention.q_lin.weight", "backbone.transformer.layer.0.attention.q_lin.bias", "backbone.transformer.layer.0.attention.k_lin.weight", "backbone.transformer.layer.0.attention.k_lin.bias", "backbone.transformer.layer.0.attention.v_lin.weight", "backbone.transformer.layer.0.attention.v_lin.bias", "backbone.transformer.layer.0.attention.out_lin.weight", "backbone.transformer.layer.0.attention.out_lin.bias", "backbone.transformer.layer.0.sa_layer_norm.weight", "backbone.transformer.layer.0.sa_layer_norm.bias", "backbone.transformer.layer.0.ffn.lin1.weight", "backbone.transformer.layer.0.ffn.lin1.bias", "backbone.transformer.layer.0.ffn.lin2.weight", "backbone.transformer.layer.0.ffn.lin2.bias", "backbone.transformer.layer.0.output_layer_norm.weight", "backbone.transformer.layer.0.output_layer_norm.bias", "backbone.transformer.layer.1.attention.q_lin.weight", "backbo

In [ ]:
def predict_text(text):

    inputs = loaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    logits = outputs.logits

    probs = torch.softmax(logits, dim=1)

    pred_id = torch.argmax(probs, dim=1).item()

    return {
        "text": text,
        "prediction": id2label[pred_id],
        "confidence": probs[0][pred_id].item(),
        "all_probs": {
            id2label[i]: round(probs[0][i].item(), 4)
            for i in range(len(id2label))
        }
    }

In [ ]:
result = predict_text("I hate all immigrants")

print(result)